# Corpus sintético de la voz propia (IureVoz, fase T4)

Genera el guion `guion_corpus.tsv` con **Qwen3-TTS 1.7B** clonando la voz del usuario (referencia: clips 037 «Órale…» y 040 «Ajá…»). Cada frase pasa por el **filtro fonético** (`filtro_fonemas.py`). Si el PER supera el 15 % o la «ll» suena como «l», se regenera con otra semilla (hasta 3 intentos) y se guarda el mejor intento.

**Entradas**: una carpeta (dataset privado de Kaggle o carpeta de Drive) con:
- `guion_corpus.tsv`, `filtro_fonemas.py` (de `bench/corpus/` en el repo)
- `ref/037.wav`, `ref/040.wav` (de `~/.local/share/iurevoz/mi-voz/`)

**Kaggle**
1. Sube la carpeta como *dataset privado* (`iurevoz-voz`).
2. Notebook nuevo → *Add data* → ese dataset.
3. Elige *Accelerator: GPU T4 x2* o *P100* e *Internet: on*.
4. **Save Version → Save & Run All**: corre en segundo plano hasta 12 h y guarda `/kaggle/working`.
5. Para continuar, agrega como entrada la *salida* de la versión anterior: el cuaderno copia lo ya generado y sigue desde ahí.

**Colab**: copia la carpeta a `MyDrive/iurevoz-voz/` y elige un entorno con GPU. La salida se escribe directo en Drive, así que si se corta la sesión basta con volver a ejecutar todo.

La voz es del usuario: el dataset y la salida deben quedarse **privados**.

In [ ]:
import os, sys, glob, shutil, time, csv
from pathlib import Path

EN_KAGGLE = Path("/kaggle").exists()
if EN_KAGGLE:
    ENTRADA = Path(next(p for p in glob.glob("/kaggle/input/*") if Path(p, "guion_corpus.tsv").exists()))
    SALIDA = Path("/kaggle/working/salida")
    PREVIAS = [Path(p) for p in glob.glob("/kaggle/input/*/salida") + glob.glob("/kaggle/input/*/*/salida")]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ENTRADA = Path("/content/drive/MyDrive/iurevoz-voz")
    SALIDA = ENTRADA / "salida"
    PREVIAS = []

MODELO = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
REFERENCIAS = {  # clip → texto exacto que se dijo
    "037": "Órale, sí, está perfecto. Mándamelo cuando puedas.",
    "040": "Ajá, entendido. Entonces lo dejamos para la próxima semana.",
}
LOTE = 4             # frases por lote; bájalo si se acaba la memoria
INTENTOS = 3         # semillas por frase si el filtro la rechaza
HORAS_MAX = 11.3     # Kaggle corta a las 12 h; se deja margen para guardar
SR = 24000

(SALIDA / "wavs").mkdir(parents=True, exist_ok=True)
for previa in PREVIAS:  # retomar: copia lo generado en versiones anteriores
    for w in (previa / "wavs").glob("*.wav"):
        if not (SALIDA / "wavs" / w.name).exists():
            shutil.copy(w, SALIDA / "wavs" / w.name)
    if (previa / "registro.csv").exists() and not (SALIDA / "registro.csv").exists():
        shutil.copy(previa / "registro.csv", SALIDA / "registro.csv")
print("entrada:", ENTRADA, "| salida:", SALIDA, "| ya generadas:", len(list((SALIDA / "wavs").glob("*.wav"))))

In [ ]:
!pip -q install qwen-tts==0.1.1 piper-phonemize==1.1.0 soundfile

In [ ]:
import numpy as np, torch, soundfile as sf
from scipy.signal import resample_poly
from qwen_tts import Qwen3TTSModel
sys.path.insert(0, str(ENTRADA))
import filtro_fonemas as ff

gpu = torch.cuda.get_device_name(0)
# T4 y P100 no tienen bf16: fp16, y si sale audio inválido, fp32 (6.8 GB, cabe en 16 GB)
def cargar(dtype):
    return Qwen3TTSModel.from_pretrained(MODELO, device_map="cuda:0", dtype=dtype)

def referencia():
    partes, textos = [], []
    for clip, texto in REFERENCIAS.items():
        x, sr = sf.read(ENTRADA / "ref" / f"{clip}.wav", dtype="float32", always_2d=True)
        partes += [resample_poly(x[:, 0], SR, sr).astype(np.float32), np.zeros(int(0.3 * SR), np.float32)]
        textos.append(texto)
    return (np.concatenate(partes), SR), " ".join(textos)

ref_audio, ref_texto = referencia()
modelo = cargar(torch.float16)
prompt = modelo.create_voice_clone_prompt(ref_audio=ref_audio, ref_text=ref_texto)
prueba, sr = modelo.generate_voice_clone(text=["Hola, esta es una prueba."], language=["Spanish"], voice_clone_prompt=prompt)
if not np.isfinite(prueba[0]).all() or np.abs(prueba[0]).max() < 1e-3:
    print("fp16 dio audio inválido: paso a fp32")
    modelo = cargar(torch.float32)
    prompt = modelo.create_voice_clone_prompt(ref_audio=ref_audio, ref_text=ref_texto)
print(gpu, "| modelo listo | sr", sr)
ff.reconocer(np.zeros(16000, np.float32), "cuda")  # carga el reconocedor de fonemas

In [ ]:
frases = []
for l in (ENTRADA / "guion_corpus.tsv").read_text(encoding="utf-8").splitlines():
    if l and not l.startswith("#"):
        i, t, origen = l.split("\t")
        frases.append((i, t))
registro = SALIDA / "registro.csv"
hechas = set()
if registro.exists():
    hechas = {r["id"] for r in csv.DictReader(registro.open(encoding="utf-8"))}
pendientes = [(i, t) for i, t in frases if i not in hechas]
print(f"{len(frases)} frases, {len(hechas)} hechas, {len(pendientes)} pendientes")

In [ ]:
def generar(textos, semilla):
    torch.manual_seed(semilla)
    wavs, sr = modelo.generate_voice_clone(text=textos, language=["Spanish"] * len(textos),
                                           voice_clone_prompt=prompt * len(textos))
    return wavs, sr

def juzgar(texto, wav, sr):
    dur = len(wav) / sr
    vel = len(texto) / max(dur, 1e-3)  # caracteres por segundo: fuera de 8–25 suele ser un error
    r = ff.evaluar(texto, resample_poly(wav, 16000, sr).astype(np.float32), "cuda")
    if not 8 <= vel <= 25:
        r["avisos"].append(f"velocidad {vel:.0f}")
    r["dur"] = dur
    return r

nuevo = not registro.exists()
fh = registro.open("a", encoding="utf-8", newline="")
w = csv.DictWriter(fh, fieldnames=["id", "texto", "semilla", "per", "avisos", "dur", "aceptada"])
if nuevo:
    w.writeheader()

inicio, n_ok, n_total, audio = time.time(), 0, 0, 0.0
for k in range(0, len(pendientes), LOTE):
    if (time.time() - inicio) / 3600 > HORAS_MAX:
        print("tiempo agotado: guarda la versión y continúa en otra"); break
    lote = pendientes[k:k + LOTE]
    mejores = {}
    restantes = lote
    for intento in range(INTENTOS):
        wavs, sr = generar([t for _, t in restantes], semilla=1000 * intento + k)
        siguen = []
        for (i, t), wav in zip(restantes, wavs):
            r = juzgar(t, wav, sr)
            if i not in mejores or r["per"] < mejores[i][1]["per"]:
                mejores[i] = (wav, r, 1000 * intento + k)
            if not ff.aceptable(r):
                siguen.append((i, t))
        restantes = siguen
        if not restantes:
            break
    for i, t in lote:
        wav, r, semilla = mejores[i]
        sf.write(SALIDA / "wavs" / f"{i}.wav", wav, sr)
        ok = ff.aceptable(r)
        w.writerow({"id": i, "texto": t, "semilla": semilla, "per": f"{r['per']:.3f}",
                    "avisos": " ".join(r["avisos"]), "dur": f"{r['dur']:.2f}", "aceptada": int(ok)})
        n_ok += ok; n_total += 1; audio += r["dur"] if ok else 0
    fh.flush()
    if (k // LOTE) % 25 == 0:
        h = (time.time() - inicio) / 3600
        print(f"{n_total} frases ({n_ok} aceptadas, {audio / 3600:.2f} h de audio) en {h:.2f} h · "
              f"faltan {len(pendientes) - n_total}", flush=True)
fh.close()

In [ ]:
import pandas as pd
df = pd.read_csv(registro, dtype={"id": str, "avisos": str})
ok = df[df.aceptada == 1]
print(f"aceptadas {len(ok)}/{len(df)} · {ok.dur.sum() / 3600:.2f} h · PER mediano {ok.per.median():.3f}")
print(df[df.aceptada == 0].avisos.fillna("PER alto").value_counts().head())
# metadata en formato LJSpeech (id|texto) solo con las aceptadas, lista para entrenar Piper
(SALIDA / "metadata.csv").write_text("".join(f"{r.id}|{r.texto}\n" for r in ok.itertuples()), encoding="utf-8")